# Ubuntu 24.04 LTS 로보틱스 개발 환경 구축 튜토리얼 (Author: jun3choi)

**머리말**
안녕하세요. 이 문서는 우분투 운영체제와 로보틱스 프로그래밍을 처음 접하는 입장에서, 학습한 내용과 환경 구축 과정을 잊지 않고 체계적으로 정리하기 위해 작성한 튜토리얼입니다. 아직 부족한 점이 많지만, 저와 같은 초보자분들이 이 문서를 참고하여 함께 차근차근 기초를 다지고 공부해 나갈 수 있기를 바랍니다. 

**환경 요약**
* **OS:** Ubuntu 24.04 LTS
* **Hardware:** 외장 그래픽카드(GPU) 탑재 시스템 기준
* **목표:** C/C++, Python 기반의 하드웨어 제어 및 ROS2, MuJoCo 시뮬레이션 환경 구축

---

## 1. 우분투 초기 설정

### 1.1 시스템 업데이트 및 필수 유틸리티
* **목적:** 갓 설치된 운영체제의 소프트웨어 저장소 목록을 최신화하고, 개발에 상시 사용되는 핵심 패키지를 설치합니다.
* **설치 항목:** `curl`, `wget` (네트워크 다운로드), `git` (버전 관리), `vim` (텍스트 편집), `build-essential` (C/C++ 컴파일러), `terminator` (화면 분할 터미널).

```bash
sudo apt update && sudo apt upgrade -y
sudo apt install terminator curl wget git vim build-essential -y
```

### 1.2 듀얼 부팅을 위한 Grub 설정
* **개념:** GRUB(GRand Unified Bootloader)은 컴퓨터 부팅 시 운영체제(Windows vs Ubuntu)를 선택하게 해주는 부트로더입니다.
* **목적:** 부팅 대기 시간을 조정하고, 2K 해상도 모니터에 최적화된 시각적 테마(Tela Theme)를 적용하여 직관적인 UI를 구성합니다.

#### 1.2.1 2K 맞춤 Tela GRUB 테마 적용 (터미널 방식)
**Step 1. 테마 다운로드**
```bash
sudo apt update && sudo apt install git -y
git clone https://github.com/vinceliuice/grub2-themes.git
cd grub2-themes
```

**Step 2. 2K 전용 설치 옵션 실행**
* `-s 2k` 파라미터를 통해 2560x1440 해상도에 맞는 아이콘 크기와 레이아웃을 지정합니다.
```bash
sudo ./install.sh -b -t tela -s 2k
```

**Step 3. GRUB 시스템 설정 파일 수정**
```bash
sudo nano /etc/default/grub
```
* 에디터가 열리면 아래와 같이 값을 수정하거나 하단에 추가합니다. (해상도 고정 및 에러 방지 목적)
```text
GRUB_DEFAULT=0
GRUB_TIMEOUT_STYLE=menu
GRUB_TIMEOUT=5
GRUB_DISTRIBUTOR=`( . /etc/os-release; echo ${NAME:-Ubuntu} ) 2>/dev/null || echo Debian`
GRUB_CMDLINE_LINUX_DEFAULT="quiet splash"
GRUB_CMDLINE_LINUX=""
GRUB_GFXMODE=2560x1440,1920x1080,auto
GRUB_GFXPAYLOAD_LINUX=keep
```
* `Ctrl + O` (저장) -> `Enter` -> `Ctrl + X` (종료)

**Step 4. 시스템 반영 및 재부팅**
```bash
sudo update-grub
reboot
```

#### 1.2.2 (선택) Grub Customizer를 통한 GUI 제어
* 터미널 환경이 익숙하지 않을 경우, 마우스 클릭으로 기본 부팅 OS를 변경할 수 있는 GUI 도구입니다.
```bash
sudo add-apt-repository ppa:danielrichter2007/grub-customizer -y
sudo apt update && sudo apt install grub-customizer -y
```

### 1.3 우분투 한글 키보드 설정
* **목적:** 초기 영문 환경에서 한글 입력 소스를 추가하여 원활한 문서 작업을 지원합니다.
1. `Settings` > `System` > `Region & Language` 이동.
2. `Manage Installed Languages` 클릭하여 누락된 한글 폰트 및 패키지 자동 설치 진행.
3. `Settings` > `Keyboard` 이동 후 `Input Sources`에서 `+` 버튼 클릭.
4. `Korean` -> `Korean (Hangul)` 선택 및 추가.
5. 우측 상단 상태 표시줄의 언어 아이콘 클릭 -> `Preferences` -> Hangul Toggle Key를 `Shift + Space` 또는 `한/영` 키로 지정.

### 1.4 개발 환경 준비
* **하드웨어 통신 권한 (dialout):** 아두이노, 센서 보드 등 USB 시리얼 통신 장비 연결 시 발생하는 'Permission Denied' 에러를 방지하기 위해 현재 사용자에게 접근 권한을 부여합니다.

```bash
# C/C++ 빌드 툴체인
sudo apt install cmake gdb -y

# 하드웨어 포트 접근 권한 부여 (실행 후 로그아웃/로그인 필요)
sudo usermod -a -G dialout $USER

# Python 기본 패키지
sudo apt install python3 python3-pip python3-venv -y

# Visual Studio Code 설치
sudo snap install code --classic

# Git 계정 연동
git config --global user.name "jun3choi"
git config --global user.email "YourEmail@example.com"
```

### 1.5 유용한 시스템 도구
* **목적:** 우분투 환경에서의 작업 효율 및 디스크 관리 편의성을 극대화합니다.

```bash
# 1. 카카오톡 구동을 위한 Wine(윈도우 호환 레이어) 설치
sudo dpkg --add-architecture i386
sudo apt update && sudo apt install wine64 wine32 -y

# 2. 디스크 관리 및 정리 도구
sudo apt install gparted ncdu -y
sudo apt autoremove -y && sudo apt clean

# 3. GNOME 데스크톱 확장 관리자 (UI 커스터마이징)
sudo apt install gnome-shell-extension-manager -y
```
* **필수 추천 확장 프로그램 (Extension Manager에서 검색 후 설치):**
  * **Clipboard Indicator:** 복사 내역 히스토리 관리 도구.
  * **Caffeine:** 장시간 빌드 시 자동 절전모드 진입 방지 도구.
  * **Just Perfection:** 상태 표시줄, 애니메이션 등 UI 요소 세밀 제어 도구.

---

## 2. Python 가상환경 설정


### 2.1 가상환경의 개념과 필요성
가상환경(Virtual Environment)은 프로젝트마다 독립된 파이썬 실행 공간을 생성하는 시스템입니다. A 프로젝트는 Python 3.8을, B 프로젝트는 Python 3.10을 요구할 때, 패키지 간의 의존성 충돌을 막고 시스템 환경을 깨끗하게 유지하기 위해 필수적으로 사용해야 합니다.

### 2.2 왜 Anaconda 대신 Miniconda/venv를 사용하는가?
데이터 사이언스 분야에서는 Anaconda가 널리 쓰이지만, **로보틱스(특히 ROS2) 환경에서는 치명적인 충돌을 유발**할 수 있으므로 가벼운 Miniconda나 venv 사용을 권장합니다.

| 구분 | Anaconda | Miniconda | Python venv |
| :--- | :--- | :--- | :--- |
| **포함 패키지** | 수백 개의 데이터/ML 패키지 기본 탑재 | Python, Conda 관리자만 탑재 | Python 표준 라이브러리 (최소) |
| **설치 용량** | 수 GB 단위 (매우 무거움) | 수백 MB 단위 (가벼움) | 수 MB (매우 가벼움) |
| **ROS2 호환성** | **매우 낮음 (경로 충돌 빈번)** | **보통 (통제 가능)** | **매우 높음 (시스템 파이썬 연동)** |

* **ROS2와 Anaconda의 충돌 원인:** ROS2는 우분투 시스템의 기본 파이썬 경로(`/usr/bin/python3`)에 강하게 의존합니다. 하지만 Anaconda를 설치하면 터미널 실행 시 기본적으로 `(base)` 환경이 켜지며 파이썬 경로를 아나콘다 내부로 덮어씌웁니다. 이 상태에서 ROS2 패키지를 빌드(`colcon build`)하면 시스템 라이브러리를 찾지 못해 빌드가 실패하는 문제가 지속적으로 발생합니다.
* **결론:** 필요한 패키지만 직접 설치하여 충돌을 막을 수 있는 **Miniconda**를 사용하거나, ROS2 노드 작성용으로는 파이썬 기본 내장 모듈인 **venv**를 사용하는 것이 가장 안정적입니다.

### 2.3 Miniconda 설치 (딥러닝/AI 연동 시 추천)
```bash
mkdir -p ~/miniconda3
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O ~/miniconda3/miniconda.sh
bash ~/miniconda3/miniconda.sh -b -u -p ~/miniconda3
rm -rf ~/miniconda3/miniconda.sh
~/miniconda3/bin/conda init bash
# 설치 완료 후 터미널 재시작 필요
```

### 2.4 venv 설치 (순수 ROS2 및 하드웨어 제어 시 추천)
```bash
# 가상환경 생성 (예: my_robot_env)
python3 -m venv ~/my_robot_env

# 가상환경 활성화 (이후 프롬프트 앞에 환경 이름이 표시됨)
source ~/my_robot_env/bin/activate

# 가상환경 비활성화 (원래 시스템 환경으로 복귀)
deactivate
```

---

## 3. ROS2 (Robot Operating System 2) 설치


### 3.1 ROS2 개요 및 도입 목적
* **개념:** ROS2는 로봇 애플리케이션 개발을 위한 오픈소스 소프트웨어 프레임워크입니다. (운영체제라기보다는 미들웨어에 가깝습니다.)
* **작동 원리 (Node & Topic):** 로봇 시스템을 하나의 거대한 프로그램으로 짜는 대신, 카메라 제어기, 모터 제어기, 센서 수집기 등을 각각의 독립된 프로그램(**Node**)으로 쪼갭니다. 이 노드들은 특정 채널(**Topic**)을 통해 데이터를 발행(**Publish**)하거나 구독(**Subscribe**)하며 실시간으로 통신합니다.
* **도입 목적:** 10채널 EMG 센서나 촉각 센서의 데이터를 취합하여 로봇 팔의 제어기로 전송하는 등, 복잡한 하드웨어 간의 비동기 통신을 매우 안정적이고 규격화된 방식으로 처리할 수 있습니다.
* **버전 선택:** Ubuntu 24.04 LTS와 공식 호환되는 최신 장기 지원(LTS) 버전인 **Jazzy Jalisco**를 설치합니다.

### 3.2 ROS2 Jazzy 설치 과정
```bash
# 1. Locale 설정 (시스템 언어를 UTF-8로 지정하여 ROS2 메시지 깨짐 방지)
sudo apt update && sudo apt install locales
sudo locale-gen en_US en_US.UTF-8
sudo update-locale LC_ALL=en_US.UTF-8 LANG=en_US.UTF-8
export LANG=en_US.UTF-8

# 2. ROS2 공식 저장소 추가 및 서명 키 등록
sudo apt install software-properties-common curl -y
sudo curl -sSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key -o /usr/share/keyrings/ros-archive-keyring.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu $(. /etc/os-release && echo $UBUNTU_CODENAME) main" | sudo tee /etc/apt/sources.list.d/ros2.list > /dev/null

# 3. ROS2 Desktop 패키지 설치 (GUI 툴 포함)
sudo apt update
sudo apt install ros-jazzy-desktop -y

# 4. 환경 변수 등록 (터미널 실행 시 ROS2 명령어 자동 로드)
echo "source /opt/ros/jazzy/setup.bash" >> ~/.bashrc
source ~/.bashrc
```

---

## 4. MuJoCo 시뮬레이터 환경 구축


### 4.1 MuJoCo 개요 및 특징
* **개념:** MuJoCo(Multi-Joint dynamics with Contact)는 구글 딥마인드에서 오픈소스로 전환한 고성능 로봇 물리 엔진 시뮬레이터입니다.
* **특징 및 활용:**
  1. **빠른 연산 속도:** 관절(Joint)과 강체(Rigid body) 간의 물리적 접촉(Contact) 연산이 타 시뮬레이터 대비 압도적으로 빠르고 안정적입니다.
  2. **딥러닝 통합:** 강화학습(RL)이나 모방학습(Imitation Learning) 모델에 로봇의 상태 및 이미지 데이터를 실시간으로 공급하는 가상 환경으로 최적화되어 있습니다.
  3. **안전성 검증:** 실제 로봇에 알고리즘을 적용하기 전, 물리법칙이 적용된 가상 세계에서 충돌과 동작을 사전 검증하여 하드웨어 파손을 막습니다. (현재 세팅된 외장 GPU 환경을 적극 활용하여 쾌적한 렌더링이 가능합니다.)

### 4.2 설치 과정
```bash
# 1. MuJoCo 설치용 숨김 디렉토리 생성
mkdir ~/.mujoco
cd ~/.mujoco

# 2. 공식 GitHub에서 Linux 빌드 파일 다운로드 및 압축 해제
wget https://github.com/google-deepmind/mujoco/releases/download/3.1.2/mujoco-3.1.2-linux-x86_64.tar.gz
tar -zxvf mujoco-3.1.2-linux-x86_64.tar.gz

# 3. Python 바인딩 모듈 설치 (파이썬 코드에서 물리 엔진을 호출하기 위함)
pip install mujoco
```